<img src="logo.png" alt="Vegeta" width="240">

# MERLIN's propulsor race — can a ducted fan win on the same battery? (27)

Notebook 26 found the pusher propeller fastest to a fire at every distance and the ducted fan far behind. This notebook
asks whether **any reasonable change** to the fan makes it win, with the **same battery** (6S 8000 mAh, the same 15 %
reserve), in two races:

- **5 km and back** — off the launch stand, climb to 150 m, dash out, turn over the point at 60° of bank, dash back; the
  clock stops over the launch site;
- **10 km** — the reach to a fire 10 km away (the readings go out by radio as soon as MERLIN is overhead), keeping the energy
  to sample for three minutes and fly home; and the 10 km out-and-back for completeness.

**Nothing is designed or solved here.** The aircraft is notebook 26's design export (`designs/data/merlin_design.json`:
masses, polar, the CFD's correction; without it the design file's defaults), and every propulsor comes from the propulsor
libraries (`designs/data/*_maps.json`): two- and three-blade propellers from notebook 25, six- and twelve-blade ones from
notebook 25c, ducted fans from notebook 25b — thrust and shaft power over airspeed × rpm, the installation on MERLIN, the
masses. A library that is missing is left out of the race. This notebook only flies them (`designs/merlin_race.py`,
`designs/merlin_flight.py`), so a thousand races take a minute or two.

What may change — the library's design space, the same for the fan's rivals:

| | design space |
|---|---|
| ducted fan | diameter 70–120 mm (rotor, hub, stators, lip and duct scaled), blade pitch 1.4–2.2 diameters, nozzle exit 65–90 % of the fan annulus, the duct's build quality (the hobby catalogue fan or a well-made one), **one, two or three stages** |
| propellers | tractor or pusher, 9–12 inch, pitch 0.6–1.2 diameters, **two, six or twelve blades** |
| both | the electrical power asked of the pack: 1900, 2700 or 3500 W (about 11, 15, 20 C), with motor and controller masses that grow with it |

The upper bound — fans whose duct loses nothing at all — is in the library too, kept apart from the reasonable designs.

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, "designs")
import merlin as mdesign
import merlin_flight as mf
import merlin_race as mr
import propulsor_maps as pm

OUT = Path("_runs/merlin_race"); OUT.mkdir(parents=True, exist_ok=True)
COLOR = {"edf": "#d62728", "tractor": "#1f77b4", "pusher": "#ff7f0e"}   # the ducted fan in red
NAME = {"edf": "ducted fan", "tractor": "tractor propeller", "pusher": "pusher propeller"}
STAGE_COLOR = {1: "#d62728", 2: "#ff9896", 3: "#8c1c13"}   # ducted fans in reds

## 1. The library

In [ ]:
LIB = pm.load()
DESIGN = mf.load_design()
ok = [e for e in LIB["entries"] if "error" not in e]
fans = [e for e in ok if e["kind"] == "edf"]
props = [e for e in ok if e["kind"] == "propeller"]
print(f"libraries found: {', '.join(LIB['kinds'])} — {len(fans)} ducted fans ({sum(e['quality'] == 'lossless' for e in fans)} of them the lossless bound), "
      f"{len(props)} propellers ({', '.join(str(b) for b in sorted({e['blades'] for e in props}))} blades)")
print("the aircraft:", "notebook 26's design export" if DESIGN else "the design file's defaults (run notebook 26 to export it)")
pd.DataFrame({kind: {k: ", ".join(f"{x:g}" for x in v) for k, v in m["space"].items()} for kind, m in LIB["meta"].items()}).fillna("")

## 2. A thousand races

Every map, as every layout it can fly in (a propeller as a tractor and as a pusher), at every power, through the three
races (`merlin_race.study`).

In [ ]:
TABLE = mr.study(LIB, design=DESIGN)
REASONABLE = TABLE[TABLE["quality"] != "lossless"]
print(f"{len(TABLE)} races; {len(REASONABLE)} of reasonable designs")
TABLE.to_csv(OUT / "races.csv", index=False)

## 3. The fastest of each kind

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 4.4))
BEST = {}
for a, race in zip(ax, mr.RACES):
    w = mr.winners(TABLE, race)
    BEST[race] = w
    for kind in ("edf", "tractor", "pusher"):
        s = w.loc[kind]
        a.plot(s.index, s[race], "o-", color=COLOR[kind], lw=2, label=NAME[kind])
    a.set(xlabel="electrical power [W]", ylabel="seconds", title=race.replace(" [s]", "")); a.legend(fontsize=8); a.grid(alpha=0.3)
fig.tight_layout()
for race, w in BEST.items():
    print(race)
    display(w[["config", race, "top speed [m/s]", "mass [kg]", "energy [Wh/km]"]].round(2))

## 4. Did any fan win?

In [ ]:
rows, any_win = {}, False
for race in mr.RACES:
    w = BEST[race]
    for P in mr.POWERS_W:
        fan, prop = w.loc[("edf", P)], min((w.loc[(k, P)] for k in ("tractor", "pusher")), key=lambda r: r[race])
        rows[(race.replace(" [s]", ""), f"{P:.0f} W")] = {"best fan [s]": fan[race], "best propeller [s]": prop[race],
                                                          "fan behind by [s]": fan[race] - prop[race], "fan": fan["config"], "propeller": prop["config"]}
        any_win |= fan[race] < prop[race]
VERDICT = pd.DataFrame(rows).T
best_fan_any = {race: REASONABLE[REASONABLE["kind"] == "edf"][race].min() for race in mr.RACES}
best_prop_low = {race: REASONABLE[(REASONABLE["kind"] != "edf") & (REASONABLE["power [W]"] == min(mr.POWERS_W))][race].min() for race in mr.RACES}
print("A reasonable ducted fan wins somewhere." if any_win else "No reasonable ducted fan wins any race at any power.")
for race in mr.RACES:
    print(f"  {race}: the best fan at any power {best_fan_any[race]:.1f} s; the best propeller at {min(mr.POWERS_W):.0f} W {best_prop_low[race]:.1f} s")
VERDICT.round(1)

## 5. What moves a fan

The fastest fan for the 5 km out-and-back at each diameter, stage count and build quality (the best pitch and nozzle of
each), at the lowest and the highest power. A bigger fan carries more air for its jet (closer to a propeller) and pays for
it in weight and nacelle; a well-made duct helps everywhere; more power helps a fan more than a propeller (its efficient
speed is higher) — and the stages?

In [ ]:
race = "5 km and back [s]"
F = REASONABLE[REASONABLE["kind"] == "edf"]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
for a, P in zip(ax, (min(mr.POWERS_W), max(mr.POWERS_W))):
    sub = F[F["power [W]"] == P]
    for (n, q), g in sub.groupby(["stages", "quality"]):
        best = g.groupby("diameter")[race].min()
        a.plot(best.index, best.values, "o-" if q == "well-made" else "s--", color=STAGE_COLOR[int(n)], lw=1.8,
               label=f"{int(n)} stage{'s' if n > 1 else ''}, {q}")
    props = REASONABLE[(REASONABLE["kind"] != "edf") & (REASONABLE["power [W]"] == P)][race].min()
    a.axhline(props, color=COLOR["pusher"], lw=1.5, ls=":", label="best propeller")
    a.set(xlabel="fan diameter [mm]", title=f"{race.replace(' [s]', '')}, {P:.0f} W"); a.grid(alpha=0.3)
ax[0].set_ylabel("seconds"); ax[1].legend(fontsize=8)
fig.tight_layout()
STAGES = F.groupby(["stages", "power [W]"])[list(mr.RACES)].min().unstack("power [W]")
display(STAGES.round(1))
PR = REASONABLE[REASONABLE["kind"] != "edf"]
BLADES = PR.groupby(["kind", "blades", "power [W]"])[list(mr.RACES)].min().unstack("power [W]")
fig, ax = plt.subplots(1, 3, figsize=(19, 4.2))
for a, race in zip(ax, mr.RACES):
    for (kind, B), g in PR.groupby(["kind", "blades"]):
        best = g.groupby("power [W]")[race].min()
        a.plot(best.index, best.values, marker="o" if kind == "pusher" else "s", ls="-" if kind == "pusher" else "--", lw=2.4 if kind == "pusher" else 1.3,
               color={2: "#1f77b4", 3: "#9467bd", 6: "#2ca02c", 12: "#ff7f0e"}[int(B)], label=f"{int(B)} blades, {kind}")
    a.set(xlabel="electrical power [W]", ylabel="seconds", title=f"propellers by blade count: {race.replace(' [s]', '')}"); a.legend(fontsize=7); a.grid(alpha=0.3)
fig.tight_layout()
BLADES.round(1)

**The blade count** (the propellers' panel and table): more blades absorb the power at a lower rpm — more static thrust;
at speed three blades give the most, twelve the least; in cruise two blades are the most efficient (notebook 25c). **The stages** (two or
three fan rings, each with its stator ring). A second or third stage adds pressure at the same flow, so the same power turns the rotors slower and
throws a faster jet — less propulsive efficiency — while adding a rotor, a stator row and duct length. In MERLIN's
races every fan is **power-limited** (its tips reach 240 m/s only far above the pack's power), so the stages lose
everywhere; they pay only for a fan held to a low tip speed, a quiet fan (notebook 25 §14).

## 6. The bound — a duct that loses nothing

The same fans with a perfectly lossless duct (no inlet, wall, nozzle or interstage loss): no fan can do better than this.

In [ ]:
rows = {}
for race in mr.RACES:
    b = mr.winners(TABLE[TABLE["quality"] == "lossless"], race, by=("kind", "power [W]"), bound=True)
    for P in mr.POWERS_W:
        prop = min(BEST[race].loc[(k, P)][race] for k in ("tractor", "pusher"))
        rows[(race.replace(" [s]", ""), f"{P:.0f} W")] = {"lossless fan [s]": b.loc[("edf", P)][race], "best propeller [s]": prop,
                                                          "lossless fan behind by [s]": b.loc[("edf", P)][race] - prop, "fan": b.loc[("edf", P)]["config"]}
BOUND = pd.DataFrame(rows).T
print("Even a lossless duct loses every race." if (BOUND["lossless fan behind by [s]"] > 0).all() else "A lossless duct would win somewhere: see the table.")
BOUND.round(1)

**Why.** A propeller makes its thrust by speeding up a lot of air a little; the fan's annulus is a tenth of a 10–12 inch
propeller's disc, so it must throw a small jet fast, and the kinetic energy left in that jet is lost: the ideal
propulsive efficiency `2 / (1 + V_jet / V)` caps it below the propeller before any loss — and no duct quality removes
that. The fan's strengths are elsewhere: an enclosed, safe and compact rotor, less sensitive to smoke and debris.

## 7. The best reasonable fan, built

The fastest reasonable fan for the 5 km out-and-back at the highest power, in MERLIN's nose (`merlin.Merlin` with its
`edf_diameter`, `edf_pitch`, `edf_exit_area_ratio`), against the baseline 90 mm fan.

In [ ]:
race = "5 km and back [s]"
top = REASONABLE[(REASONABLE["kind"] == "edf") & (REASONABLE["power [W]"] == max(mr.POWERS_W))].sort_values(race).iloc[0]
e = LIB["by_id"][top["id"]]
P_BEST = dict(propulsion="edf", edf_diameter=e["diameter_mm"], edf_pitch=e["pitch_ratio"] * e["diameter_mm"], edf_exit_area_ratio=e["exit_area_ratio"])
print(f"{top['config']}: {top[race]:.1f} s, top speed {top['top speed [m/s]']:.1f} m/s, {top['mass [kg]']:.2f} kg, {int(e['stages'])} stage(s)")
fig, ax = plt.subplots(2, 1, figsize=(13, 5.2), sharex=True)
for a, (p, title) in zip(ax, ((dict(propulsion="edf"), "baseline: 90 mm fan (notebook 26)"), (P_BEST, f"fastest fan: {top['config']}"))):
    for poly in mdesign.outline(p)["side"]:
        a.fill(poly[:, 0], poly[:, 1], color=COLOR["edf"], alpha=0.35, ec="k", lw=0.6)
    a.set_aspect("equal"); a.set_title(title, fontsize=10); a.grid(alpha=0.3)
ax[-1].set_xlabel("x [m]")
fig.tight_layout()
try:
    g = mdesign.Merlin().generate(**P_BEST)
    print("CAD:", "valid" if g.shape.isValid() else "INVALID", f"{len(g.shape.Solids())} solid, bounding box {np.round(g.dimensions, 0)} mm")
except Exception as exc:
    print("CAD not built here:", exc)

## 8. Export

In [ ]:
doc = {"source": "27_merlin_race", "libraries": {k: str(v) for k, v in pm.LIBRARIES.items()}, "powers_W": list(mr.POWERS_W),
       "winners": {race: w[["config", race, "top speed [m/s]", "mass [kg]"]].reset_index().to_dict("records") for race, w in BEST.items()},
       "verdict": VERDICT.reset_index().to_dict("records"), "bound": BOUND.reset_index().to_dict("records"),
       "best_fan": {"id": top["id"], "merlin": P_BEST}}
(OUT / "merlin_race.json").write_text(json.dumps(doc, indent=1, default=float))
print("written", OUT / "merlin_race.json", "and", OUT / "races.csv")